In [1]:
# Importo las librerías para conectarme y trabajar con dataframes
from sqlalchemy import create_engine
import pandas as pd
import pyodbc
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv


# Cargo las variables de conexión desde el archivo .env (override por si el kernel ya tenía valores cacheados)
load_dotenv(override=True)

True

In [2]:
# Configuro la conexión a PostgreSQL de origen usando las variables del .env
usuario = os.getenv("PG_USUARIO")
password = os.getenv("PG_PASSWORD")
servidor = os.getenv("PG_SERVIDOR")
puerto = os.getenv("PG_PUERTO")
base_datos = os.getenv("PG_BASE_DATOS")

cadena = (
    f"postgresql+psycopg2://{usuario}:{password}@"
    f"{servidor}:{puerto}/{base_datos}"
)

engine_origen = create_engine(cadena)

try:
    with engine_origen.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [3]:
# Extraigo los datos de productos desde la base de datos origen
consulta_producto = "SELECT * FROM tienda.producto"
df_producto = pd.read_sql_query(consulta_producto, engine_origen)

In [4]:
# Configuro la conexión al stage en SQL Server usando las variables del .env
servidor_sqlserver = os.getenv("SQLSERVER_SERVIDOR")
usuario_sqlserver = os.getenv("SQLSERVER_USUARIO")
password_sqlserver = os.getenv("SQLSERVER_PASSWORD")
base_datos_stage = os.getenv("SQLSERVER_BASE_DATOS")
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")
driver_sqlserver = os.getenv("SQLSERVER_DRIVER")

cadena_sqlserver = (
    f"mssql+pyodbc://{usuario_sqlserver}:{quote_plus(password_sqlserver)}@"
    f"{servidor_sqlserver}/{base_datos_stage}?driver={quote_plus(driver_sqlserver)}"
)

engine_stage = create_engine(cadena_sqlserver)
try:
    with engine_stage.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [5]:
# Agrego la fecha y hora de carga al dataframe de productos
df_producto["fecha_carga"] = pd.Timestamp.now()

In [6]:
# Reviso una muestra de productos antes de cargarla
df_producto.head()

,producto_id,codigo,nombre,descripcion,categoria_id,subcategoria_id,franquicia_id,marca_id,proveedor_id,precio,costo,stock,stock_minimo,activo,fecha_creacion,fecha_modificacion,fecha_carga
0,1,FRK-0001,Figuras articulada The Legend of Zelda - Modelo 1,Artículo oficial inspirado en The Legend of Zelda,1,1,11,2,3,190.0,121.60,85,5,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:13:18.327656
1,2,FRK-0002,Figuras articulada Pokémon - Modelo 2,Artículo oficial inspirado en Pokémon,1,1,4,2,4,420.0,260.40,34,8,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:13:18.327656
2,3,FRK-0003,Figuras articulada El Señor de los Anillos - M...,Artículo oficial inspirado en El Señor de los ...,1,1,9,2,7,380.0,224.20,84,12,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:13:18.327656
3,4,FRK-0004,Funko Pop One Piece - Modelo 1,Artículo oficial inspirado en One Piece,1,2,7,1,1,140.0,92.40,26,12,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:13:18.327656
4,5,FRK-0005,Funko Pop Dragon Ball - Modelo 2,Artículo oficial inspirado en Dragon Ball,1,2,5,1,8,225.0,137.25,27,5,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:13:18.327656


In [7]:
# Cargo los productos en la tabla del stage
df_producto.to_sql(
    name="producto",
    con=engine_stage,
    schema=esquema_stage,
    if_exists="append",
    index=False
)

72

In [8]:
# Verifico el contenido de productos en el stage
consulta_stage = f"SELECT * FROM {esquema_stage}.producto"
df_producto_stage = pd.read_sql_query(consulta_stage, engine_stage)
df_producto_stage.head()

,producto_id,codigo,nombre,descripcion,categoria_id,subcategoria_id,franquicia_id,marca_id,proveedor_id,precio,costo,stock,stock_minimo,activo,fecha_creacion,fecha_modificacion,fecha_carga
0,1,FRK-0001,Figuras articulada The Legend of Zelda - Modelo 1,Artículo oficial inspirado en The Legend of Zelda,1,1,11,2,3,190.0,121.60,85,5,True,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:13:18.327
1,2,FRK-0002,Figuras articulada Pokémon - Modelo 2,Artículo oficial inspirado en Pokémon,1,1,4,2,4,420.0,260.40,34,8,True,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:13:18.327
2,3,FRK-0003,Figuras articulada El Señor de los Anillos - M...,Artículo oficial inspirado en El Señor de los ...,1,1,9,2,7,380.0,224.20,84,12,True,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:13:18.327
3,4,FRK-0004,Funko Pop One Piece - Modelo 1,Artículo oficial inspirado en One Piece,1,2,7,1,1,140.0,92.40,26,12,True,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:13:18.327
4,5,FRK-0005,Funko Pop Dragon Ball - Modelo 2,Artículo oficial inspirado en Dragon Ball,1,2,5,1,8,225.0,137.25,27,5,True,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:13:18.327
